# LeetCode #1132 — Reported Posts II (SQL)

**Medium · Database / SQL · SQLite thực hành bằng Python 3, MySQL để submit**  
[Đề gốc LeetCode](https://leetcode.com/problems/reported-posts-ii/) · [Đối chiếu đề và bảng ví dụ](https://leetcode.doocs.org/en/lc/1132/)

**Chú ý:** đây là bài **SQL**, không phải DSA Python. `sqlite3` giúp chạy truy vấn và nhìn kết quả ngay trong Jupyter; trên LeetCode hãy nộp **câu SQL**, không phải `class Solution`.

**Mục tiêu:** hiểu `WHERE`, `DISTINCT`, `LEFT JOIN`, `GROUP BY`, `COUNT`, `AVG` và `ROUND` thông qua dữ liệu thực, rồi tự xây được truy vấn hoàn chỉnh.

## 1. Mục tiêu học

1. Phân biệt **hành động `report` vì `spam`** với `view`, `like`, báo cáo vì lý do khác.
2. Đếm **bài đăng khác nhau**, không đếm số lượt báo cáo hay số người báo cáo.
3. Hiểu kết nối `Actions.post_id = Removals.post_id`.
4. Tính **tỷ lệ phần trăm theo từng ngày**, rồi lấy **trung bình các ngày** (không lấy tỷ lệ chung của mọi bài).
5. Viết CTE và truy vấn MySQL đúng tên cột `average_daily_percent`.
6. Tự test các ca báo cáo trùng, ngày không có spam, bài không bị xóa.

## 2. Đề bài và lược đồ

**Bảng `Actions`** gồm: `user_id` (số người dùng), `post_id` (mã bài), `action_date` (ngày), `action` (`view`, `report`, ...), `extra` (ví dụ lý do báo cáo là `spam`). **Có thể có hàng trùng**.

**Bảng `Removals`** gồm: `post_id` (khóa chính, không trùng) và `remove_date` (ngày bị gỡ).

Hãy tìm **trung bình của tỷ lệ phần trăm theo ngày** các bài đăng **đã bị gỡ** trong số các bài đăng **bị báo cáo spam** ngày đó; làm tròn **hai chữ số thập phân**. Chỉ những ngày có ít nhất một báo cáo spam mới đóng góp vào trung bình.

**Rất quan trọng:** đề nói không cần quan tâm giá trị `remove_date` trong phép tính; chỉ cần biết `post_id` có xuất hiện trong bảng `Removals` hay không. Tránh tự thêm điều kiện ngày ngoài đề.

## 3. Ví dụ gốc — vì sao đáp án là 75.00?

Hai ngày có báo cáo spam:

| Ngày | Các bài bị report spam (không trùng) | Đã có trong Removals | Tỷ lệ |
|---|---|---|---:|
| 2019-07-02 | `{3}` | `{3}` | `100%` |
| 2019-07-04 | `{2,4}` | `{2}` | `50%` |

Ngày `2019-07-03` chỉ có báo cáo `racism`, nên **không tính**. Kết quả là `(100 + 50) / 2 = 75.00`, không lấy tổng số bài đã gỡ chia tổng số bài bị báo cáo của mọi ngày.

## 4. Nền tảng: `sqlite3` và hai bảng trong RAM

`sqlite3.connect(':memory:')` tạo cơ sở dữ liệu tạm để học ngay trong notebook. `cursor.execute(SQL)` thực thi câu lệnh SQL; `fetchall()` trả các hàng kết quả về Python dưới dạng danh sách `tuple`.

Với bài này, SQL dùng khi **nộp** giống logic SQL dùng khi **thực hành**; riêng kiểu dữ liệu `DATE` có thể được lưu dưới dạng chuỗi ISO `YYYY-MM-DD` trong SQLite.

In [1]:
import sqlite3
conn=sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE Actions(user_id INTEGER, post_id INTEGER, action_date TEXT, action TEXT, extra TEXT);
CREATE TABLE Removals(post_id INTEGER PRIMARY KEY, remove_date TEXT);
""")
print('Đã tạo bảng Actions và Removals trong bộ nhớ')

Đã tạo bảng Actions và Removals trong bộ nhớ


## 5. Nạp dữ liệu mẫu đúng theo đề

`None` trong Python tương ứng `NULL` trong SQL. Dùng `executemany` để chèn nhiều hàng an toàn, không ghép tay chuỗi SQL từ dữ liệu người dùng.

In [2]:
ACTIONS_SAMPLE=[
    (1,1,'2019-07-01','view',None),
    (1,1,'2019-07-01','like',None),
    (1,1,'2019-07-01','share',None),
    (2,2,'2019-07-04','view',None),
    (2,2,'2019-07-04','report','spam'),
    (3,4,'2019-07-04','view',None),
    (3,4,'2019-07-04','report','spam'),
    (4,3,'2019-07-02','view',None),
    (4,3,'2019-07-02','report','spam'),
    (5,2,'2019-07-03','view',None),
    (5,2,'2019-07-03','report','racism'),
    (5,5,'2019-07-03','view',None),
    (5,5,'2019-07-03','report','racism'),
]
REMOVALS_SAMPLE=[(2,'2019-07-20'),(3,'2019-07-18')]
conn.executemany('INSERT INTO Actions VALUES (?,?,?,?,?)', ACTIONS_SAMPLE)
conn.executemany('INSERT INTO Removals VALUES (?,?)', REMOVALS_SAMPLE)
assert conn.execute('SELECT COUNT(*) FROM Actions').fetchone()[0]==13
assert conn.execute('SELECT COUNT(*) FROM Removals').fetchone()[0]==2
print('Actions: 13 hàng; Removals: 2 hàng')

Actions: 13 hàng; Removals: 2 hàng


## 6. `SELECT`, `FROM`, `WHERE` đọc thế nào?

```sql
SELECT post_id, action_date
FROM Actions
WHERE action = 'report' AND extra = 'spam';
```

Đọc là: "Trong bảng Actions, chỉ lấy các hàng có **hành động báo cáo** và **lý do spam**; hiện mã bài và ngày". Hai điều kiện nối bằng `AND` nghĩa là **phải thỏa cả hai**.

In [3]:
q_filter="""
SELECT post_id, action_date
FROM Actions
WHERE action = 'report' AND extra = 'spam'
ORDER BY action_date,post_id;
"""
rows=conn.execute(q_filter).fetchall()
print('Sau WHERE:',rows)
assert rows==[(3,'2019-07-02'),(2,'2019-07-04'),(4,'2019-07-04')]

Sau WHERE: [(3, '2019-07-02'), (2, '2019-07-04'), (4, '2019-07-04')]


## 7. `DISTINCT` — khác nhau là khác theo *cặp* (ngày, bài)

Một bài có thể được **nhiều người báo cáo trong cùng một ngày**. Đề yêu cầu đếm **bài đăng** được báo cáo, không đếm hàng hay số người. Vì vậy ta lấy cặp `(action_date,post_id)` khác nhau.

Không nên `DISTINCT post_id` *toàn cục* vì cùng một bài bị report spam vào hai ngày khác nhau thì mỗi ngày vẫn phải được tính riêng.

In [4]:
demo_duplicate=ACTIONS_SAMPLE+[(9,2,'2019-07-04','report','spam'),(9,2,'2019-07-04','report','spam')]
print('Số report spam (hàng, gồm trùng):',sum(a[3]=='report' and a[4]=='spam' for a in demo_duplicate))
distinct_pairs={(a[2],a[1]) for a in demo_duplicate if a[3]=='report' and a[4]=='spam'}
print('Cặp (ngày, bài) khác nhau:',sorted(distinct_pairs))
assert len(distinct_pairs)==3

Số report spam (hàng, gồm trùng): 5
Cặp (ngày, bài) khác nhau: [('2019-07-02', 3), ('2019-07-04', 2), ('2019-07-04', 4)]


## 8. CTE (Common Table Expression) là gì?

`WITH spam_reports AS (...) SELECT ... FROM spam_reports` đặt tên cho một kết quả trung gian. Bạn có thể hiểu nó giống "bảng tạm trong phạm vi một câu SQL". Hãy tách riêng bước lọc và loại trùng, sau đó mới làm những phép tính khó hơn.

In [5]:
SQL_SPAM_REPORTS="""
WITH spam_reports AS (
    SELECT DISTINCT action_date,post_id
    FROM Actions
    WHERE action = 'report' AND extra = 'spam'
)
SELECT action_date, post_id FROM spam_reports
ORDER BY action_date,post_id;
"""
rows=conn.execute(SQL_SPAM_REPORTS).fetchall()
print(rows)
assert rows==[('2019-07-02',3),('2019-07-04',2),('2019-07-04',4)]

[('2019-07-02', 3), ('2019-07-04', 2), ('2019-07-04', 4)]


## 9. `LEFT JOIN` — giữ cả bài *không bị gỡ*

`LEFT JOIN Removals r ON s.post_id = r.post_id`: mỗi bài trong `spam_reports` luôn có mặt; nếu mã bài chưa nằm trong `Removals` thì cột `r.post_id` là `NULL`.

**Vì sao không dùng `INNER JOIN`?** Vì inner join sẽ loại bỏ bài không bị gỡ, làm biến mất **mẫu số**, khiến tỷ lệ dễ bị tính thành 100% sai.

In [6]:
SQL_JOIN="""
WITH spam_reports AS (
    SELECT DISTINCT action_date,post_id
    FROM Actions
    WHERE action='report' AND extra='spam'
)
SELECT s.action_date,s.post_id,r.post_id AS removed_post_id
FROM spam_reports s
LEFT JOIN Removals r ON s.post_id=r.post_id
ORDER BY s.action_date,s.post_id;
"""
rows=conn.execute(SQL_JOIN).fetchall()
for row in rows:print(row)
assert rows==[('2019-07-02',3,3),('2019-07-04',2,2),('2019-07-04',4,None)]

('2019-07-02', 3, 3)
('2019-07-04', 2, 2)
('2019-07-04', 4, None)


## 10. `NULL` khác `0` và khác chuỗi rỗng

Trong SQL, `NULL` nghĩa là **không có giá trị khớp / chưa biết**. `WHERE r.post_id = NULL` không phải cách hỏi null; phải dùng `IS NULL`. Trong bài này, `r.post_id IS NOT NULL` nghĩa là bài báo cáo đó xuất hiện trong bảng gỡ bỏ.

In [7]:
print('Không bị gỡ:',conn.execute("""
    WITH s AS (SELECT DISTINCT action_date,post_id FROM Actions WHERE action='report' AND extra='spam')
    SELECT s.post_id FROM s LEFT JOIN Removals r ON s.post_id=r.post_id
    WHERE r.post_id IS NULL
""").fetchall())
assert conn.execute('SELECT COUNT(*) FROM Removals WHERE post_id IS NULL').fetchone()[0]==0

Không bị gỡ: [(4,)]


## 11. `GROUP BY action_date` — mỗi ngày thành một hàng

Trong mỗi nhóm ngày, sau khi đã loại trùng, ta tính:

\[
percent(day)=100\times \frac{\text{số post đã gỡ trong ngày}}{\text{số post report spam trong ngày}}.
\]

`COUNT(*)` đếm tất cả bài trong nhóm (sau `DISTINCT`). `COUNT(r.post_id)` chỉ đếm giá trị **khác NULL**, tức bài đã gỡ. `100.0` giúp tránh nhầm với chia nguyên ở một số hệ cơ sở dữ liệu.

In [8]:
SQL_DAILY="""
WITH spam_reports AS (
    SELECT DISTINCT action_date,post_id
    FROM Actions
    WHERE action='report' AND extra='spam'
)
SELECT s.action_date,
       COUNT(*) AS reported_posts,
       COUNT(r.post_id) AS removed_posts,
       100.0*COUNT(r.post_id)/COUNT(*) AS day_percent
FROM spam_reports s
LEFT JOIN Removals r ON s.post_id=r.post_id
GROUP BY s.action_date
ORDER BY s.action_date;
"""
rows=conn.execute(SQL_DAILY).fetchall()
for row in rows:print(row)
assert rows==[('2019-07-02',1,1,100.0),('2019-07-04',2,1,50.0)]

('2019-07-02', 1, 1, 100.0)
('2019-07-04', 2, 1, 50.0)


## 12. `AVG` — *trung bình tỷ lệ của các ngày*, không phải chia gộp

Từ bảng tỷ lệ ngày `[100,50]`, cần `(100+50)/2=75`.

Nếu chia gộp theo tất cả bài: `2/3*100=66.67%` — **sai**, vì đề cho trọng số **bằng nhau giữa các ngày**, không phải giữa các bài.

In [9]:
daily_rates=[100.0,50.0]
average=sum(daily_rates)/len(daily_rates)
pooled=(1+1)/(1+2)*100
print('Đúng: AVG(rate mỗi ngày) =',average)
print('Sai: tỷ lệ gộp =',pooled)
assert average==75 and abs(pooled-66.6666666667)<1e-6

Đúng: AVG(rate mỗi ngày) = 75.0
Sai: tỷ lệ gộp = 66.66666666666666


## 13. `ROUND(x, 2)` — làm tròn **sau** khi lấy trung bình

Không làm tròn từng ngày trước khi lấy `AVG` nếu đề không yêu cầu. Chỉ áp dụng `ROUND(AVG(day_percent),2)` ở cuối cùng.

Tên cột kết quả bắt buộc là **`average_daily_percent`**.

In [10]:
MAIN_SQL="""
WITH spam_reports AS (
    SELECT DISTINCT action_date, post_id
    FROM Actions
    WHERE action = 'report' AND extra = 'spam'
), daily AS (
    SELECT s.action_date,
           100.0 * COUNT(r.post_id) / COUNT(*) AS day_percent
    FROM spam_reports s
    LEFT JOIN Removals r ON s.post_id = r.post_id
    GROUP BY s.action_date
)
SELECT ROUND(AVG(day_percent), 2) AS average_daily_percent
FROM daily;
"""
answer=conn.execute(MAIN_SQL).fetchone()[0]
print('average_daily_percent =',answer)
assert answer==75.0

average_daily_percent = 75.0


## 14. Bản SQL nộp LeetCode (MySQL)

Trong editor MySQL, dán **chỉ đoạn SQL** sau. Nó không phụ thuộc Python. `WITH` và `LEFT JOIN` đều có trong MySQL 8. Bài này **không có** `class Solution`.

```sql
WITH spam_reports AS (
    SELECT DISTINCT action_date, post_id
    FROM Actions
    WHERE action = 'report' AND extra = 'spam'
), daily AS (
    SELECT s.action_date,
           100.0 * COUNT(r.post_id) / COUNT(*) AS day_percent
    FROM spam_reports s
    LEFT JOIN Removals r ON s.post_id = r.post_id
    GROUP BY s.action_date
)
SELECT ROUND(AVG(day_percent), 2) AS average_daily_percent
FROM daily;
```

Nhờ CTE đầu tiên loại trùng theo **ngày và post**, việc `COUNT(r.post_id)` ở CTE thứ hai là chính xác do `Removals.post_id` là khóa chính.

## 15. Trace toàn bộ pipeline bằng 5 câu hỏi

| Bước | Câu hỏi | Kết quả ví dụ |
|---|---|---|
| WHERE | Đâu là `report` vì `spam`? | ba hàng ở hai ngày |
| DISTINCT | Một bài trong một ngày tính mấy lần? | đúng một lần |
| LEFT JOIN | Bài nào được gỡ? | `3`, `2`; không phải `4` |
| GROUP BY | Mỗi ngày có tỷ lệ gì? | 100%, 50% |
| AVG+ROUND | Trung bình tỷ lệ các ngày? | 75.00 |

Đừng gộp các bước khi còn chưa rõ mẫu số của phần trăm.

In [11]:
filtered=conn.execute(SQL_SPAM_REPORTS).fetchall()
joined=conn.execute(SQL_JOIN).fetchall()
grouped=conn.execute(SQL_DAILY).fetchall()
print('1. DISTINCT:',filtered)
print('2. LEFT JOIN:',joined)
print('3. GROUP BY:',grouped)
print('4. FINAL:',conn.execute(MAIN_SQL).fetchall())

1. DISTINCT: [('2019-07-02', 3), ('2019-07-04', 2), ('2019-07-04', 4)]
2. LEFT JOIN: [('2019-07-02', 3, 3), ('2019-07-04', 2, 2), ('2019-07-04', 4, None)]
3. GROUP BY: [('2019-07-02', 1, 1, 100.0), ('2019-07-04', 2, 1, 50.0)]
4. FINAL: [(75.0,)]


## 16. Brute Force bằng Python — viết oracle độc lập SQL

Tạo `dict` ánh xạ `ngày -> set(post_id)` để loại trùng tự nhiên. Sau đó so sánh từng `post_id` với `set` các bài đã gỡ. Đây **không phải code nộp LeetCode**, chỉ để nhìn rõ nghĩa của câu hỏi và kiểm chứng SQL.

In [12]:
from collections import defaultdict

def python_oracle(actions,removals):
    by_day=defaultdict(set)
    for user_id,post_id,action_date,action,extra in actions:
        if action=='report' and extra=='spam':
            by_day[action_date].add(post_id)
    removed={post_id for post_id,remove_date in removals}
    if not by_day:
        return None
    rates=[]
    for day, posts in by_day.items():
        rates.append(100*len(posts & removed)/len(posts))
    return round(sum(rates)/len(rates),2)

assert python_oracle(ACTIONS_SAMPLE,REMOVALS_SAMPLE)==75.0
print('Oracle Python:',python_oracle(ACTIONS_SAMPLE,REMOVALS_SAMPLE))

Oracle Python: 75.0


## 17. Hàm chạy test trong database mới cho mỗi ca

Để bộ kiểm thử không phụ thuộc dữ liệu từ test trước, mỗi ca được dựng trong một SQLite `:memory:` riêng, chèn dữ liệu, thực thi SQL và đóng kết nối. Đây là cách tránh *trạng thái ngầm* thường làm notebook khó kiểm thử.

In [13]:
def run_case(actions,removals,query=MAIN_SQL):
    db=sqlite3.connect(':memory:')
    try:
        db.executescript("""
        CREATE TABLE Actions(user_id INTEGER,post_id INTEGER,action_date TEXT,action TEXT,extra TEXT);
        CREATE TABLE Removals(post_id INTEGER PRIMARY KEY,remove_date TEXT);
        """)
        db.executemany('INSERT INTO Actions VALUES (?,?,?,?,?)',actions)
        db.executemany('INSERT INTO Removals VALUES (?,?)',removals)
        return db.execute(query).fetchone()[0]
    finally:
        db.close()

assert run_case(ACTIONS_SAMPLE,REMOVALS_SAMPLE)==75.0

## 18. Test 1 — nhiều người report **cùng một bài** trong cùng ngày

Ví dụ 1 ngày: bài `10` bị ba report spam nhưng đã gỡ; bài `11` bị một report spam và chưa gỡ. Tỷ lệ đúng `1/2 = 50%`, **không phải** `3/4 = 75%`.

In [14]:
A=[
  (1,10,'2020-01-01','report','spam'),
  (2,10,'2020-01-01','report','spam'),
  (2,10,'2020-01-01','report','spam'),
  (3,11,'2020-01-01','report','spam'),
]
R=[(10,'2020-01-03')]
assert run_case(A,R)==python_oracle(A,R)==50.0
print('Báo cáo trùng -> 50%: PASS')

Báo cáo trùng -> 50%: PASS


## 19. Test 2 — ngày không spam, và bài spam không bị gỡ

Ngày chỉ có `report` nhưng `extra='racism'` không tính. Nếu một ngày có spam nhưng **không bài nào bị gỡ**, ngày đó có tỷ lệ 0% và **vẫn phải tính vào trung bình**.

In [15]:
A=[
  (1,1,'2020-01-01','report','spam'),
  (2,2,'2020-01-02','report','spam'),
  (3,3,'2020-01-03','report','racism'),
  (4,4,'2020-01-03','view',None),
]
R=[(1,'2020-01-04')]
assert run_case(A,R)==python_oracle(A,R)==50.0  # (100% + 0%)/2
assert run_case([(1,5,'2020-01-01','report','racism')],[]) is None
print('Ngày 0%, ngày không spam: PASS')

Ngày 0%, ngày không spam: PASS


## 20. Test 3 — cùng bài bị spam vào **hai ngày khác nhau**

Một bài có thể xuất hiện trong cả hai tập theo ngày. Không được `DISTINCT post_id` trước khi phân nhóm ngày: làm vậy sẽ mất một đóng góp hợp lệ.

In [16]:
A=[
  (1,7,'2020-01-01','report','spam'),
  (2,7,'2020-01-02','report','spam'),
  (3,8,'2020-01-02','report','spam'),
]
R=[(7,'2020-01-20')]
# Ngày 1: 100%; ngày 2: 50% → trung bình 75%
assert run_case(A,R)==python_oracle(A,R)==75.0

## 21. Test 4 — dữ liệu ngẫu nhiên, đối chiếu Python

Mỗi ca có nhiều hành động, người dùng, ngày, nguyên nhân báo cáo. Oracle Python tách biệt với câu SQL giúp bắt lỗi `DISTINCT`, `GROUP BY`, và `LEFT JOIN`. Với phần thập phân sát ngưỡng làm tròn, so dung sai nhỏ thay vì đòi biểu diễn số thực giống từng bit.

In [17]:
import random
rng=random.Random(1132)
for case_id in range(180):
    actions=[]
    for _ in range(rng.randint(1,35)):
        post_id=rng.randint(1,8)
        day=f'2020-01-{rng.randint(1,5):02d}'
        action=rng.choice(['view','report','report','share'])
        extra=rng.choice(['spam','racism',None])
        actions.append((rng.randint(1,5),post_id,day,action,extra))
    removals=[(p,'2020-03-01') for p in range(1,9) if rng.randint(0,1)]
    actual=run_case(actions,removals)
    expected=python_oracle(actions,removals)
    assert (actual is None and expected is None) or (actual is not None and expected is not None and abs(actual-expected)<0.011), (case_id,actual,expected)
print('180 test SQLite so với Oracle: PASS')

180 test SQLite so với Oracle: PASS


## 22. Giải thích vì sao truy vấn SQL đúng

1. CTE `spam_reports`: mỗi phần tử `(day,post)` tương ứng đúng một bài khác nhau bị báo cáo spam vào ngày đó.
2. `LEFT JOIN Removals`: do bảng Removals có `post_id` duy nhất, mỗi phần tử bên trái cho đúng một hàng sau join, và `r.post_id` khác `NULL` đúng khi bài được gỡ.
3. `GROUP BY day`: `COUNT(r.post_id)/COUNT(*)` chính xác là tỷ lệ bài đã gỡ trong các bài spam của *ngày đó*.
4. `AVG(day_percent)` coi mỗi ngày có trọng số bằng nhau; `ROUND(...,2)` làm tròn đầu ra theo yêu cầu.

Như vậy truy vấn trả đúng một số theo định nghĩa của đề.

## 23. Phân tích hiệu năng (phụ thuộc hệ CSDL)

Gọi `A` số hàng Actions, `S` số cặp `(ngày,post)` spam khác nhau, `R` số hàng Removals.

- Lọc `Actions`: quét có thể `O(A)` nếu không có index.
- `DISTINCT` và `GROUP BY`: có thể tốn sort/hashing, chi phí phụ thuộc query planner và index; không khẳng định tuyệt đối `O(A)` cho mọi MySQL/SQLite.
- Join theo `Removals.post_id` (primary key) có thể tra cứu hiệu quả bằng index.

Bài SQL nên phân tích **kế hoạch thực thi** và index thực tế, thay vì gán Big-O đơn giản cho cả câu lệnh.

## 24. Những lỗi thường gặp: sai → lý do → sửa

| Lỗi | Hậu quả | Sửa |
|---|---|---|
| `COUNT(*)` trên Actions chưa loại trùng | Nhiều report cùng bài bị đếm nhiều lần | `SELECT DISTINCT action_date, post_id` |
| `INNER JOIN Removals` | Mất bài chưa gỡ | `LEFT JOIN` |
| `COUNT(r.post_id) / COUNT(*)` rồi gộp tất cả ngày | Tỷ lệ toàn cục, không phải trung bình ngày | CTE `daily` và `AVG` |
| Chỉ `WHERE extra='spam'` | Có thể nhận cả hành động không phải `report` | Thêm `action='report'` |
| Gán điều kiện `remove_date >= action_date` | Thêm luật ngoài đề | Chỉ join theo `post_id` |
| Làm tròn từng ngày | Có thể sai chữ số cuối | `ROUND(AVG(day_percent),2)` |

In [18]:
WRONG_INNER="""
WITH s AS (SELECT DISTINCT action_date,post_id FROM Actions WHERE action='report' AND extra='spam')
SELECT 100.0*COUNT(r.post_id)/COUNT(*)
FROM s INNER JOIN Removals r ON s.post_id=r.post_id;
"""
wrong=run_case(ACTIONS_SAMPLE,REMOVALS_SAMPLE,WRONG_INNER)
print('INNER JOIN sai ->',wrong,'| câu SQL đúng ->',run_case(ACTIONS_SAMPLE,REMOVALS_SAMPLE))
assert wrong==100.0

INNER JOIN sai -> 100.0 | câu SQL đúng -> 75.0


## 25. Mẹo nhận diện pattern SQL

Hễ đề hỏi **tỷ lệ theo nhóm rồi lấy trung bình trên các nhóm**, hãy hỏi:

1. **Đơn vị đếm** là hàng, người dùng, hay mã bài *khác nhau*?
2. **Mẫu số** có giữ đủ trường hợp không xảy ra sự kiện (dùng LEFT JOIN)?
3. Lấy `AVG` trên **tỷ lệ từng nhóm** hay muốn **tỷ lệ gộp**?

Đây là ba câu hỏi tổng quát dùng lại với retention rate, conversion rate và thống kê theo ngày.

## 26. Biến thể — xem tỷ lệ từng ngày thay vì chỉ một số

Trong phân tích kinh doanh, bảng mỗi ngày dễ đọc hơn. Truy vấn `SQL_DAILY` đã giải được biến thể này với các cột `reported_posts`, `removed_posts`, `day_percent`. Trên LeetCode #1132, vẫn phải trả **chỉ một cột** `average_daily_percent`.

In [19]:
daily=conn.execute(SQL_DAILY).fetchall()
assert len(daily)==2
assert [row[3] for row in daily]==[100.0,50.0]
print('Bảng biến thể theo ngày:',daily)

Bảng biến thể theo ngày: [('2019-07-02', 1, 1, 100.0), ('2019-07-04', 2, 1, 50.0)]


## 27. Bài tập tự luyện — 5 câu tăng dần

1. **Tính tay:** 3 ngày có tỷ lệ 100%, 0%, 50%; trung bình bằng bao nhiêu?
2. **Dự đoán output:** `COUNT(*)` và `COUNT(r.post_id)` khác nhau thế nào sau `LEFT JOIN`?
3. **TODO:** Viết câu `SELECT` lấy `DISTINCT action_date,post_id` khi `action='report'` và `extra='spam'`.
4. **TODO:** Viết truy vấn hiển thị tỷ lệ `day_percent` theo ngày, chưa cần trung bình.
5. **Mini challenge:** Sửa truy vấn sai dùng `INNER JOIN` và tính tỷ lệ gộp, thay bằng CTE + `LEFT JOIN` + `AVG`.

Lưu câu SQL vào string; không thực thi TODO chưa hoàn thiện.

In [20]:
EX_1132_3 = """-- TODO: SELECT DISTINCT ... FROM Actions WHERE ..."""
EX_1132_4 = """-- TODO: WITH s AS (...) SELECT action_date, ... GROUP BY ..."""
EX_1132_5 = """-- TODO: CTE lọc -> tỷ lệ từng ngày -> ROUND(AVG(...),2)"""
# Sau khi làm xong:
# assert conn.execute(EX_1132_3).fetchall()  # chạy test tự viết

## 28. Gợi ý theo ba mức

| Bài | Mức 1 — hỏi đúng | Mức 2 — lệnh SQL | Mức 3 — gần code |
|---|---|---|---|
| 1 | Mỗi ngày có một trọng số | `sum/len` | `(100+0+50)/3` |
| 2 | NULL không được COUNT(column) | `COUNT(*)` vs `COUNT(r.post_id)` | 3 bài, 2 bài gỡ → `(3,2)` |
| 3 | Loại trùng theo ngày+bài | `SELECT DISTINCT` | `WHERE action='report' AND extra='spam'` |
| 4 | Giữ cả bài chưa gỡ | `LEFT JOIN`, `GROUP BY` | `100.0*COUNT(r.post_id)/COUNT(*)` |
| 5 | Trung bình các ngày | `WITH ... daily ...` | `ROUND(AVG(day_percent),2)` |

## 29. Đáp án tham khảo — chỉ xem sau bài tập

- Bài 1: `(100 + 0 + 50) / 3 = 50%`.
- Bài 2: Với `3` bài sau `LEFT JOIN`, trong đó `2` bài có `r.post_id` thì `COUNT(*)=3`, `COUNT(r.post_id)=2`.
- Bài 3–5: truy vấn SQL tham khảo trong cell dưới. Bài 5 là câu submit của #1132.

In [21]:
ANSWER_1132_3="""
SELECT DISTINCT action_date,post_id
FROM Actions
WHERE action='report' AND extra='spam';
"""
ANSWER_1132_4=SQL_DAILY
ANSWER_1132_5=MAIN_SQL
assert len(conn.execute(ANSWER_1132_3).fetchall())==3
assert [r[3] for r in conn.execute(ANSWER_1132_4).fetchall()]==[100.0,50.0]
assert conn.execute(ANSWER_1132_5).fetchone()[0]==75.0
print('Đáp án SQL tham khảo: PASS')

Đáp án SQL tham khảo: PASS


## 30. Checklist trước khi submit SQL

- [ ] Đúng hai bảng `Actions`, `Removals` và khóa `post_id`.
- [ ] Chỉ lọc `action='report' AND extra='spam'`.
- [ ] `DISTINCT` theo cả `action_date,post_id`.
- [ ] `LEFT JOIN`, không bỏ bài chưa gỡ.
- [ ] Phần trăm **mỗi ngày** trước, `AVG` các ngày sau.
- [ ] `ROUND(...,2)` ở cuối và đặt alias `average_daily_percent`.
- [ ] Không thêm điều kiện `remove_date` ngoài yêu cầu đề.
- [ ] Nộp câu truy vấn **MySQL**, không nộp Python hay SQLite setup.